# Tabular Classification Project: MLP with Embeddings

## Purpose

This is a **self-directed project** focused on building a neural network for tabular classification using PyTorch embeddings.

**What you'll practice:**
- Data exploration and preprocessing for real-world messy data
- Handling categorical features with learned embeddings in PyTorch
- Building a full NN pipeline: DataLoaders, training loop, evaluation
- Regularization: dropout, batch normalization, weight decay
- Proper evaluation with imbalanced data (beyond accuracy)

**Why neural networks on tabular data?**

Trees dominate most tabular benchmarks, but NNs have real advantages:
- Learned embeddings capture relationships between categories that label encoding misses
- Scale better with dataset size (300K+ rows)
- Can combine with other modalities (images, text) in the same model
- Transfer learning potential — pretrained embeddings can bootstrap new tasks

If you've also done the trees project on the same dataset, compare results at the end. The goal isn't to "beat" trees — it's to understand when each approach makes sense.

In [ ]:
# Colab setup - run this cell first if you're on Google Colab
try:
    import google.colab
    print("Colab environment ready! All required packages are pre-installed.")
except ImportError:
    pass  # Not on Colab, no action needed

## Dataset Options

Below are tabular datasets suitable for this project. The key requirement: **mix of categorical and numerical features** so you practice building embeddings.

### Summary Table

| Dataset | Rows | Cat:Num | Task | Imbalance | Difficulty |
|---------|------|---------|------|-----------|------------|
| **Bank Marketing** | 41,188 | 10:10 | Binary | 88/12 | Medium |
| **Adult Income** | 48,842 | 8:6 | Binary | 76/24 | Medium |
| **Census-Income KDD** | 299,285 | ~24:16 | Binary | 94/6 | Hard |
| **Secondary Mushroom** | 61,069 | 17:3 | Binary | ~balanced | Medium |
| **Porto Seguro** | 595,212 | 14+17bin:26 | Binary | 96/4 | Hard |

## Dataset Deep Dive

### 1. Bank Marketing (Portuguese Bank)

**What it is:** Predict whether a client will subscribe to a term deposit after a phone marketing campaign.

**Source:** UCI ML Repository. Data from a Portuguese bank's direct marketing campaigns (2008-2013).

**Download:** [UCI ML Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing) — use `bank-additional-full.csv` (semicolon-delimited)

| Pros | Cons |
|------|------|
| Good size (41K rows) - NNs can compete | Semicolon delimiter (minor) |
| Balanced mix: 10 categorical, 10 numerical | `duration` feature leaks info (should be dropped) |
| Rich categoricals (job has 12 levels, month, day_of_week) | |
| Missing values coded as "unknown" - forces real decisions | |
| 88/12 class imbalance - teaches evaluation beyond accuracy | |
| Includes economic context features (euribor rate, employment) | |
| Interpretable domain | |

**Categorical features:** job (12), marital (4), education (8), default, housing, loan, contact, month, day_of_week, poutcome
**Numerical features:** age, duration, campaign, pdays, previous, emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed

**Expected performance:** F1 ~0.5-0.6 on minority class, AUC ~0.90+

**Best for:** First project. Good categorical variety, interpretable features, real messiness without being overwhelming.

### 2. Adult Income (Census Income)

**What it is:** Predict whether a person earns >$50K/year based on census data.

**Source:** UCI ML Repository, extracted from 1994 Census database.

**Download:** [UCI ML Repository](https://archive.ics.uci.edu/dataset/2/adult)

| Pros | Cons |
|------|------|
| Good size (48K rows) | Overused in tutorials |
| High-cardinality categoricals (occupation, country) | Older data (1994) |
| Missing values marked with "?" | |
| 76/24 class imbalance | |
| Well-benchmarked - easy to sanity-check results | |

**Categorical features:** workclass, education, marital-status, occupation, relationship, race, sex, native-country
**Numerical features:** age, fnlwgt, education-num, capital-gain, capital-loss, hours-per-week

**Expected performance:** Accuracy 85-87%, both approaches competitive.

**Best for:** Safe pick. You'll find reference implementations everywhere to compare against.

### 3. Census-Income KDD

**What it is:** Same domain as Adult Income but 6x the data and more features. Predict income >$50K.

**Source:** UCI ML Repository / KDD.

**Download:** [UCI ML Repository](https://archive.ics.uci.edu/dataset/117/census+income+kdd) — no header row, column names in `.names` file

| Pros | Cons |
|------|------|
| Massive (300K rows) - NNs genuinely benefit from scale | No header row, need to parse column names from docs |
| ~40 features with rich categorical variety | Lots of "Not in universe" values to handle |
| 94/6 class imbalance - forces serious evaluation strategy | More preprocessing overhead |
| Pre-split train/test | Feature redundancy to manage |

**Expected performance:** Similar domain to Adult but more severe imbalance changes the challenge.

**Best for:** When you want Adult-like data at NN-competitive scale. This is where NNs start showing real advantage.

### 4. Secondary Mushroom

**What it is:** Predict whether a mushroom is edible or poisonous based on physical characteristics.

**Source:** UCI ML Repository (2021 version - much better than the classic 1987 version).

| Pros | Cons |
|------|------|
| 61K rows, solid size | Nearly balanced classes (less evaluation challenge) |
| Overwhelmingly categorical: 17 of 20 features | Anonymized somewhat |
| Interpretable (cap-shape, gill-color, habitat) | |
| Missing values present | |
| Can visualize what embeddings learn about mushroom features | |

**Categorical features:** cap-shape, cap-surface, cap-color, does-bruise-bleed, gill-attachment, gill-spacing, gill-color, stem-root, stem-surface, stem-color, veil-type, veil-color, has-ring, ring-type, spore-print-color, habitat, season
**Numerical features:** cap-diameter, stem-height, stem-width

**Expected performance:** High accuracy achievable (90%+).

**Best for:** Maximum embedding practice. 17 categoricals means you'll get very comfortable building embedding layers.

### 5. Porto Seguro (Safe Driver Prediction)

**What it is:** Predict whether a driver will file an insurance claim. From a Kaggle competition (5,200 teams).

**Source:** Kaggle competition (Porto Seguro's Safe Driver Prediction).

| Pros | Cons |
|------|------|
| Massive (595K rows) | Anonymized features (can't interpret what embeddings learn) |
| Explicitly labeled categoricals (`_cat` suffix) | Brutal 96/4 imbalance |
| Heavy missingness (coded as -1) | Larger preprocessing effort |
| Well-benchmarked from competition | Anonymized features limit interpretability |
| NNs placed well in competition | |

**Expected performance:** Normalized Gini ~0.29 (top Kaggle solutions).

**Best for:** Challenge project after you've got the pipeline working on something simpler.

## Recommendations

### Top Pick: Bank Marketing

**Why:**
- 10 categorical features with real variety (job has 12 levels, education has 8)
- 10 numerical features including economic indicators
- "unknown" values force real missing-data decisions
- 88/12 imbalance makes evaluation meaningful (accuracy alone is misleading)
- Interpretable domain — you can explain what embeddings learn
- `duration` feature is a known data leak — teaches the lesson of thinking before modeling

### Second Pick: Adult Income

**Why:**
- Already downloaded locally, well-benchmarked everywhere
- High-cardinality categoricals (native-country ~41 values, occupation ~14)
- Slightly less interesting domain but tons of reference implementations to compare against

### For a Challenge: Census-Income KDD or Porto Seguro

**Why:**
- 300K-600K rows — this is where NNs genuinely outperform trees
- More severe imbalance forces you to go beyond basic metrics
- Heavier preprocessing — closer to real-world data engineering

### Skip for This Project

| Dataset | Why Skip |
|---------|----------|
| Titanic | Too small (891 rows) — NNs will overfit, trees win trivially |
| Heart Disease | Too small (303 rows) |
| Wine Quality | No categoricals — misses the entire embedding practice |
| California Housing | Regression + no categoricals |
| Secondary Mushroom | Good for pure embedding practice but nearly balanced classes reduce evaluation challenge |

## Your Choice

**Dataset I'm using for this project:**

*(Write your choice here)*

# Part A: Neural Networks (MLP with Embeddings)

MLPs can compete with tree-based models on tabular data, especially when:
- Dataset is large (>10K rows)
- High-cardinality categorical features benefit from learned embeddings
- You want to combine with other modalities later

This is the primary focus of this project.

## MLP Tasks

### Phase A1: Data Exploration

**Task A1.1: Load and inspect the data**
- Load the dataset (CSV, sklearn, or other source)
- Check shape, dtypes, first few rows
- Identify target variable

**Task A1.2: Explore distributions**
- Check for missing values
- Examine target class balance
- Look at numerical feature distributions
- Count unique values in categorical columns

**Task A1.3: Basic visualizations**
- Histograms of numerical features
- Bar charts of categorical features
- Correlation heatmap (numerical only)

### Phase A2: Preprocessing

**Task A2.1: Investigate data leakage**
- The `duration` feature (call length in seconds) is a known leak in this dataset
- Think about it: can you know call duration *before* you make the call?
- Check the correlation between `duration` and the target — what do you see?
- Read the dataset authors' note about this in `bank-additional-names.txt` (line 56)
- Look into: what is "target leakage" vs "train-test contamination"? They're different failure modes
- Decide what to do with `duration` and document your reasoning

**Task A2.2: Handle missing values**
- Decide strategy: drop, impute with median/mode, or create "Unknown" category
- Implement your chosen strategy

**Task A2.3: Split data**
- Train/validation/test split (e.g., 80/10/10)
- Use stratification for imbalanced data
- Do this BEFORE encoding/scaling to avoid data leakage

**Task A2.4: Identify feature types**
- List categorical columns and numerical columns
- Store cardinality of each categorical (needed for embeddings)

**Task A2.5: Encode categorical features**
- Label encoding (integers 0 to n-1)
- Embeddings will learn representations, no one-hot needed

**Task A2.6: Scale numerical features**
- StandardScaler is essential for neural networks
- Fit on train, transform train/val/test
- Keep numerical and categorical arrays separate

### Phase A3: Data Loading (PyTorch)

**Task A3.1: Create TabularDataset class**
- Subclass `torch.utils.data.Dataset`
- Store numerical features as FloatTensor
- Store categorical features as LongTensor (for embeddings)
- Store labels
- Implement `__len__` and `__getitem__`

**Task A3.2: Create DataLoaders**
- Training loader with shuffle=True
- Validation/test loaders with shuffle=False
- Choose batch size (start with 256 or 512)

### Phase A4: Model Architecture

**Task A4.1: Define TabularNN class**
- Subclass `nn.Module`
- Create embedding layers for each categorical feature
- Embedding dimension rule: `min(50, (n_categories + 1) // 2)`

**Task A4.2: Define forward pass**
- Get embeddings for each categorical
- Concatenate: [numerical | all embedded categoricals]
- Pass through MLP layers:
  - Linear → BatchNorm → ReLU → Dropout (repeat 2-3x)
  - Final Linear for output

**Task A4.3: Verify architecture**
- Create model instance, move to GPU
- Pass dummy batch, check output shape
- Print parameter count

### Phase A5: Training

**Task A5.1: Set up training components**
- Loss: `BCEWithLogitsLoss` (binary) or `CrossEntropyLoss` (multi-class)
- Handle class imbalance with `pos_weight`
- Optimizer: Adam with `weight_decay` for L2 regularization
- Scheduler: `ReduceLROnPlateau`

**Task A5.2: Write training loop**
- For each epoch:
  - Training phase: forward, loss, backward, update
  - Validation phase: compute metrics without gradients
- Track loss and metrics history

**Task A5.3: Implement early stopping**
- Save best model based on validation metric
- Stop if no improvement for N epochs

**Task A5.4: Monitor training**
- Print progress every few epochs
- Watch for overfitting (train improves, val doesn't)

### Phase A6: Evaluation

**Task A6.1: Compute metrics**
- Load best model
- Accuracy, ROC-AUC, classification report (precision, recall, F1)
- Store predictions for later comparison

**Task A6.2: Plot learning curves**
- Training loss vs epochs
- Validation metric vs epochs

**Task A6.3: Inspect embeddings**
- Examine what the embedding layers learned
- Do similar categories cluster together?

### Choose a dataset and just start!